<a href="https://colab.research.google.com/github/Esbern/Sankey-diagrams/blob/main//Sankey_03/sankey3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [91]:
import requests
import pandas as pd
import plotly.graph_objects as go
import duckdb
import string
import plotly.colors

In [92]:
# Airtable credentials
api_key = 'patwjsizhgQyQkZkT.f9e8b1595df5b527d0d01d3a45af0dfa77eab63707e18398ad62f1f3818a9ce9'
base_id = 'apprKfEKZ2Ju74g9w'

In [93]:
# Helper function to fetch data from Airtable
def fetch_airtable_data(table_id):
    url = f"https://api.airtable.com/v0/{base_id}/{table_id}"
    headers = {"Authorization": f"Bearer {api_key}"}
    records = []
    params = {}

    while True:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code != 200:
            raise Exception(f"Failed to fetch data: {response.text}")
        data = response.json()
        records.extend([record["fields"] | {"id": record["id"]} for record in data["records"]])
        if "offset" in data:
            params["offset"] = data["offset"]
        else:
            break

    return pd.DataFrame(records)

In [94]:
# Fetch data
tabel0 = fetch_airtable_data("tblzHR1WHYHA5MlwQ")  # Policy Source
tabel1 = fetch_airtable_data("tbl7OYOXduME11uh7")  # Targets (mellemtabel)
tabel2 = fetch_airtable_data("tblVarbVYd96JUE6f")  # Target Group
tabel3 = fetch_airtable_data("tblTRyuT48bBN24QG")  # Land uses

In [95]:
# Sørg for at 'Territorial reference point' er liste (og ikke NaN)
tabel0['Territorial reference point'] = tabel0['Territorial reference point'].apply(lambda x: x if isinstance(x, list) else [])

# Explode 'Territorial reference point'
tabel0_exploded = tabel0.explode('Territorial reference point')

# Fjern tomme værdier
tabel0_exploded = tabel0_exploded.dropna(subset=['Territorial reference point'])

# Gør det samme for 'Targets'
tabel0_exploded['Targets (policy targets)'] = tabel0_exploded['Targets (policy targets)'].apply(lambda x: x if isinstance(x, list) else [])

# Explode 'Targets'
tabel0_exploded = tabel0_exploded.explode('Targets (policy targets)')

# Join med Targets-tabel
merged_pt = tabel0_exploded.merge(
    tabel1.rename(columns={'id': 'Targets (policy targets)'}),
    on='Targets (policy targets)',
    suffixes=('', '_Targets (policy targets)')
)
    # Sørg for at 'Target Group' er liste
merged_pt['Target Group'] = merged_pt['Target Group'].apply(lambda x: x if isinstance(x, list) else [])

# Explode – én række per tilknyttet Target Group
merged_pt = merged_pt.explode('Target Group')

#  Fjern eksisterende kolonnen "Target Group" for at undgå konflikt
merged_pt_clean = merged_pt.drop(columns=['Target Group'])


# Join med tabel2 (Target Group)
merged_ptg = merged_pt.merge(
    tabel2.rename(columns={'id': 'Target Group ID'}),
    left_on='Target Group',
    right_on='Target Group ID',
    suffixes=('', '_tg')
)

#  Sørg for at 'Land uses' er en liste
tabel1['Land uses'] = tabel1['Land uses'].apply(lambda x: x if isinstance(x, list) else [])

#  Explode Targets – én række per Land Use
targets_exploded = tabel1.explode('Land uses')

# Join med Land Uses for at få navn på land use
targets_lu = targets_exploded.merge(
    tabel3.rename(columns={'id': 'Land Use ID'}),
    left_on='Land uses',
    right_on='Land Use ID',
    suffixes=('', '_lu')
)

# Omdøb 'Targets' til 'Target ID' så vi kan matche senere
targets_lu_renamed = targets_lu.rename(columns={'Targets': 'Target ID'})

# Sørg for at 'Target ID' er en liste
targets_lu_renamed['Target ID'] = targets_lu_renamed['Target ID'].apply(lambda x: x if isinstance(x, list) else [])

# Explode så vi har én række pr. Target ID
targets_lu_renamed = targets_lu_renamed.explode('Target ID')


lu_target = targets_lu_renamed[['Target ID', 'Name']].dropna()

In [96]:
#  Territorial reference point → Target
territory_target = tabel0_exploded[['Territorial reference point', 'Targets (policy targets)']].dropna()
territory_target.columns = ['Territorial reference point', 'Target ID']

# Genopbyg merged_ptg_renamed (Target Group + Target ID)
merged_ptg_renamed = merged_ptg.rename(columns={'Targets (policy targets)': 'Target ID'})

# Target → Target Group
tg_target = merged_ptg_renamed[['Target Group', 'Target ID']].dropna()

#  Target → Land Use
lu_target = targets_lu_renamed[['Target ID', 'Name']].dropna()

#  Merge på Target ID
merged_flow = territory_target.merge(tg_target, on='Target ID', how='inner') \
                              .merge(lu_target, on='Target ID', how='inner')

#  Fjern Target ID – vi viser det ikke
merged_flow = merged_flow[['Territorial reference point', 'Target Group', 'Name']]

#  Drop rækker med manglende værdier
merged_flow = merged_flow.dropna()



In [97]:
# Træk alle links i flyderækkefølge
df_sankey = pd.concat([
    merged_flow[['Territorial reference point', 'Target Group']].rename(columns={
        'Territorial reference point': 'source',
        'Target Group': 'target'
    }),
    merged_flow[['Target Group', 'Name']].rename(columns={
        'Target Group': 'source',
        'Name': 'target'
    })
])

In [99]:
# Liste over alle unikke noder
unique_labels = pd.unique(df_sankey[['source', 'target']].values.ravel())
label_to_index = {label: i for i, label in enumerate(unique_labels)}

# Koder til sankey
df_sankey['source_index'] = df_sankey['source'].map(label_to_index)
df_sankey['target_index'] = df_sankey['target'].map(label_to_index)
df_sankey['value'] = 1  # eller brug faktisk værdi hvis relevant

# Brug Plotly-paletten
node_colors = plotly.colors.qualitative.Plotly
color_map = {label: node_colors[i % len(node_colors)] for i, label in enumerate(unique_labels)}
node_colors_list = [color_map[label] for label in unique_labels]

# Funktion til at lysne farver
def lighten(hex_color, factor=0.5):
    from plotly.colors import hex_to_rgb
    r, g, b = hex_to_rgb(hex_color)
    r = int(r + (255 - r) * factor)
    g = int(g + (255 - g) * factor)
    b = int(b + (255 - b) * factor)
    return f'rgb({r},{g},{b})'

# Lys farve til links ud fra source
link_colors = [lighten(node_colors_list[src], factor=0.8) for src in df_sankey['source_index']]

In [ ]:
# Unik liste over alle noder
all_labels = pd.unique(df_sankey[['source', 'target']].values.ravel())
label_to_index = {label: i for i, label in enumerate(all_labels)}

# Omsæt labels til numeriske koder
source_indices = df_sankey['source'].map(label_to_index)
target_indices = df_sankey['target'].map(label_to_index)

# Antag én forbindelse pr. række
values = [1] * len(df_sankey)

# Lav Sankey
fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=60,
        thickness=20,
        line=dict(color="lightgray", width=0.5),
        label=list(unique_labels),
        color=node_colors_list
    ),
    link=dict(
        source=df_sankey['source_index'],
        target=df_sankey['target_index'],
        value=df_sankey['value'],
        color=link_colors
    )
)])

fig.update_layout(title_text="Territorial Reference Point → Target Group → Land Use", font_size=12, height=800)
fig.show()

#Gammelt


In [ ]:
# Register in DuckDB
duckdb.register("tabel0", t0)
duckdb.register("tabel1_exp", t1)
duckdb.register("tabel2_exp", t2)
duckdb.register("tabel3", tabel3)

In [ ]:
# SQL query
query = """
SELECT
    t0."Policy source"       AS policy_source,
    t1."Target name"         AS mid_target,
    t2."Target Group"        AS target_group,
    t3."Target name"         AS final_target
FROM
    tabel0 t0
JOIN
    tabel1_exp t1 ON t0.target_id = t1.id
JOIN
    tabel2_exp t2 ON t1.target_group_id = t2.id
JOIN
    tabel3 t3 ON t2.target_id = t3.id
"""

results = duckdb.sql(query).df()

In [ ]:
# Trin 1: forkort og saml labels
results['final_target'] = results['final_target'].apply(lambda x: x[:40] + '…' if isinstance(x, str) and len(x) > 50 else x)
source_labels = results['policy_source']
# Lav mapping fra policy source label til A, B, C ...
policy_source_labels = pd.unique(source_labels)
abc_labels = list(string.ascii_uppercase)[:len(policy_source_labels)]
policy_label_map = dict(zip(policy_source_labels, abc_labels))

# Udskift policy source labels i alle lister (kun i source_labels)
source_labels_abc = source_labels.map(policy_label_map)

# Sæt også forklaring til senere brug under diagrammet
policy_label_explanation = {abc: orig for orig, abc in policy_label_map.items()}

middle_labels = results['target_group']
# Target group mapping: 1, 2, 3, ...
target_group_labels = pd.unique(middle_labels)
number_labels = [str(i+1) for i in range(len(target_group_labels))]
target_group_label_map = dict(zip(target_group_labels, number_labels))
middle_labels_num = middle_labels.map(target_group_label_map)
target_group_label_explanation = {num: orig for orig, num in target_group_label_map.items()}

target_labels = results['final_target']
# all_labels = pd.concat([source_labels_abc, middle_labels, target_labels])
all_labels = pd.concat([source_labels_abc, middle_labels_num, target_labels])
unique_labels = pd.unique(all_labels)
label_to_index = {label: i for i, label in enumerate(unique_labels)}

In [ ]:
# Trin 2: forbindelser
links1 = pd.DataFrame({
    'source': source_labels_abc.map(label_to_index),
    'target': middle_labels_num.map(label_to_index),
    'value': 1
})
links2 = pd.DataFrame({
    'source': middle_labels_num.map(label_to_index),
    'target': target_labels.map(label_to_index),
    'value': 1
})

all_links = pd.concat([links1, links2])

In [ ]:
# Brug f.eks. Plotlys palette
node_colors = plotly.colors.qualitative.Plotly

# Tildel farver til unikke labels (noder)
color_map = {label: node_colors[i % len(node_colors)] for i, label in enumerate(unique_labels)}

# Liste med farver i samme rækkefølge som unique_labels
node_colors_list = [color_map[label] for label in unique_labels]


# Funktion til at lysne farver (uden brug af gennemsigtighed)
def lighten(hex_color, factor=0.5):
    from plotly.colors import hex_to_rgb
    r, g, b = hex_to_rgb(hex_color)
    r = int(r + (255 - r) * factor)
    g = int(g + (255 - g) * factor)
    b = int(b + (255 - b) * factor)
    return f'rgb({r},{g},{b})'

# Lysnet farve til hver link ud fra source-node
link_colors = [lighten(node_colors_list[src], factor=0.8) for src in all_links['source']]



# Trin 3: Sankey-diagram
fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=60,
        thickness=20,
        line=dict(color="lightgray", width=0.5),
        label=list(unique_labels),
        color=node_colors_list
    ),
    link=dict(
        source=all_links['source'],
        target=all_links['target'],
        value=all_links['value'],
        color=link_colors
    )
)])
fig.update_layout(title_text="Policy Source → Target Group → Target", font_size=12, height=6000)
fig.show()

print("Forklaring på labels i Sankey-diagrammet:\n")

print("Policy Source (A, B, C...):")
for abc, orig in policy_label_explanation.items():
    print(f"  {abc}: {orig}")

print("\nTarget Group (1, 2, 3...):")
for num, orig in target_group_label_explanation.items():
    print(f"  {num}: {orig}")

In [ ]:
# 1. Lav arbejdskopier
labels_list = list(unique_labels)
links_df = all_links.copy()

# 2. Find noder i sidste kolonne: noder der aldrig er source
all_targets = set(links_df['target'])
all_sources = set(links_df['source'])
last_node_indices = list(all_targets - all_sources)

# 3. Fjern links til/fra disse noder
mask = links_df['source'].isin(last_node_indices) | links_df['target'].isin(last_node_indices)
filtered_links = links_df[~mask]

# 4. Fjern noderne fra labels
filtered_labels = [label for i, label in enumerate(labels_list) if i not in last_node_indices]

# 5. Opbyg mapping fra gamle til nye index
old_to_new_index = {
    old_i: new_i for new_i, (old_i, _) in enumerate([
        (i, label) for i, label in enumerate(labels_list) if i not in last_node_indices
    ])
}

# 6. Opdater source og target
filtered_links = filtered_links.copy()  # undgå warnings
filtered_links['source'] = filtered_links['source'].map(old_to_new_index)
filtered_links['target'] = filtered_links['target'].map(old_to_new_index)

# 7. Filtrer node-farver
filtered_node_colors = [node_colors_list[i] for i in range(len(labels_list)) if i not in last_node_indices]

# 8. Filtrer link-farver (kun hvis det er array-lignende)
try:
    filtered_link_colors = link_colors[~mask]
except TypeError:
    filtered_link_colors = link_colors  # hvis det ikke er subscriptable

# 9. Lav nyt Sankey-diagram med samme stil
fig2 = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=60,
        thickness=20,
        line=dict(color="lightgray", width=0.5),
        label=filtered_labels,
        color=filtered_node_colors
    ),
    link=dict(
        source=filtered_links['source'],
        target=filtered_links['target'],
        value=filtered_links['value'],
        color=filtered_link_colors
    )
)])

# 10. Samme layout som før
fig2.update_layout(
    title_text="Policy Source → Target Group",
    font_size=12,
    height=6000
)

fig2.show()


In [ ]:
# 10. Gem som interaktiv HTML-fil
fig2.write_html("sankey_diagram_filtered.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Filteret kode med original labels

In [ ]:
#  1. Gem original labels hvis ikke allerede gemt
if 'final_target_original' not in results.columns:
    results['final_target_original'] = results['final_target']

#  2. Genskab labels uden forkortelser eller mappings
original_policy_labels = results['policy_source']
original_group_labels = results['target_group']
original_target_labels = results['final_target_original']

all_original_labels = pd.concat([original_policy_labels, original_group_labels, original_target_labels])
original_labels = pd.unique(all_original_labels).tolist()

#  3. Filtrér noder uden udgående links
labels_list = list(original_labels)
links_df = all_links.copy()

all_targets = set(links_df['target'])
all_sources = set(links_df['source'])
last_node_indices = list(all_targets - all_sources)

mask = links_df['source'].isin(last_node_indices) | links_df['target'].isin(last_node_indices)
filtered_links = links_df[~mask]

#  4. Fjern noder og genmap index
filtered_labels = [label for i, label in enumerate(labels_list) if i not in last_node_indices]

old_to_new_index = {
    old_i: new_i for new_i, (old_i, _) in enumerate([
        (i, label) for i, label in enumerate(labels_list) if i not in last_node_indices
    ])
}

filtered_links = filtered_links.copy()
filtered_links['source'] = filtered_links['source'].map(old_to_new_index)
filtered_links['target'] = filtered_links['target'].map(old_to_new_index)

#  5. Filtrér farver
filtered_node_colors = [node_colors_list[i] for i in range(len(labels_list)) if i not in last_node_indices]
try:
    filtered_link_colors = link_colors[~mask]
except TypeError:
    filtered_link_colors = link_colors

#  6. Lav nyt diagram
fig_original_names = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=60,
        thickness=20,
        line=dict(color="lightgray", width=0.5),
        label=filtered_labels,
        color=filtered_node_colors
    ),
    link=dict(
        source=filtered_links['source'],
        target=filtered_links['target'],
        value=filtered_links['value'],
        color=filtered_link_colors
    )
)])

fig_original_names.update_layout(
    title_text="Policy Source → Target Group → Target (uden sidste kolonne, med oprindelige labels)",
    font_size=12,
    height=2000
)

# 7. Gem som interaktiv HTML og download
#fig_original_names.write_html("sankey_original_labels_filtered.html")

#from google.colab import files
#files.download("sankey_original_labels_filtered.html")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>